# Experiment 15 - LightGBM Baseline

This experiment tests LightGBM as a different boosting model from the current best XGBoost model.

Current Kaggle benchmark: **0.941680**

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier

PROJECT_ROOT = Path(r"C:\Users\aakif\Documents\DataCompetition")
TRAIN_PATH = PROJECT_ROOT / "data" / "train.csv"

train = pd.read_csv(TRAIN_PATH)

X = train.drop(columns=["Will_Buy_EV", "id"])
y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Data loaded successfully")
print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

Data loaded successfully
Training rows: 534932
Validation rows: 133733


In [2]:
model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=0.0,
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("LightGBM model created.")

LightGBM model created.


In [3]:
pipeline.fit(X_train, y_train)

valid_predictions = pipeline.predict_proba(X_valid)[:, 1]
roc_auc = roc_auc_score(y_valid, valid_predictions)

previous_best = 0.941776
kaggle_benchmark = 0.941680
difference_validation = roc_auc - previous_best
difference_kaggle = roc_auc - kaggle_benchmark

print("=" * 60)
print("EXPERIMENT 14 RESULTS")
print("=" * 60)
print(f"LightGBM ROC-AUC: {roc_auc:.6f}")
print(f"Previous local best: {previous_best:.6f}")
print(f"Difference vs local best: {difference_validation:+.6f}")
print(f"Kaggle Submission 05: {kaggle_benchmark:.6f}")
print(f"Difference vs Kaggle benchmark: {difference_kaggle:+.6f}")

if roc_auc > previous_best:
    print("\nNEW LOCAL BEST MODEL")
else:
    print("\nDid not beat the current local best model.")

EXPERIMENT 14 RESULTS
LightGBM ROC-AUC: 0.941493
Previous local best: 0.941776
Difference vs local best: -0.000283
Kaggle Submission 05: 0.941680
Difference vs Kaggle benchmark: -0.000187

Did not beat the current local best model.
